# 🎨 Fooocus Ultra-Realista (Sin Censura) - Google Colab

Este notebook ejecuta **Fooocus** con los mejores modelos SDXL ultra-realistas y libres de censura:
- 🧠 **Soporte Google Drive:** Copia tus modelos y LoRAs guardados en `MyDrive/RuinedFooocus` de forma segura (Anti-RAM).
- 🌐 **Descarga Automática de Modelos:** Elige abajo qué modelos ultra-realistas quieres tener disponibles si no los tienes en Drive.
- 🚫 **Sin Censura / NSFW Activo:** Todos los filtros de censura están deshabilitados.
- 💾 **Salidas a Drive:** Las imágenes se guardan automáticamente en tu Google Drive.

In [ ]:
# @title 🚀 Configuración y Lanzamiento de Fooocus
# @markdown ### 📥 Selecciona qué modelos ultra-realistas deseas descargar si no están en tu Drive:
Descargar_CyberRealistic_XL = True #@param {type:"boolean"}
Descargar_Juggernaut_XL_v9 = False #@param {type:"boolean"}
Descargar_RealVisXL_v5 = False #@param {type:"boolean"}
Descargar_Loras_Realismo = True #@param {type:"boolean"}

import os
import sys
from tqdm import tqdm

# 1. Repositorio
%cd /content
if not os.path.exists('/content/Fooocus'):
    print("\n📦 Descargando tu versión optimizada de Fooocus...")
    !git clone https://github.com/Christianebg1/Fooocus.git
    %cd /content/Fooocus
else:
    %cd /content/Fooocus
    !git pull

local_checkpoints = '/content/Fooocus/models/checkpoints'
local_loras = '/content/Fooocus/models/loras'
os.makedirs(local_checkpoints, exist_ok=True)
os.makedirs(local_loras, exist_ok=True)

# 2. Conectar Google Drive (si está disponible)
drive_base = '/content/drive/MyDrive/RuinedFooocus'
drive_checkpoints = os.path.join(drive_base, 'checkpoints')
drive_loras = os.path.join(drive_base, 'loras')
drive_outputs = os.path.join(drive_base, 'outputs')

drive_connected = False
try:
    from google.colab import drive
    print("\n1️⃣ Conectando Google Drive...")
    drive.mount('/content/drive')
    if os.path.exists('/content/drive/MyDrive'):
        drive_connected = True
        os.makedirs(drive_outputs, exist_ok=True)
        !rm -rf /content/Fooocus/outputs
        !ln -s "{drive_outputs}" /content/Fooocus/outputs
        print("✅ Google Drive vinculado. Las imágenes se guardarán en MyDrive/RuinedFooocus/outputs")
except Exception as e:
    print("ℹ️ Google Drive no conectado. Las imágenes se guardarán temporalmente en Colab.")

# 3. Copia segura Anti-RAM desde Drive
def copiar_archivos_sin_ram(origen, destino):
    if not os.path.exists(origen):
        return 0
    copiados = 0
    for archivo in os.listdir(origen):
        ruta_origen = os.path.join(origen, archivo)
        ruta_destino = os.path.join(destino, archivo)
        if os.path.isfile(ruta_origen) and not os.path.exists(ruta_destino):
            peso_total = os.path.getsize(ruta_origen)
            print(f"\n📥 Copiando desde Drive: {archivo}")
            with open(ruta_origen, 'rb') as fsrc, open(ruta_destino, 'wb') as fdst, tqdm(
                total=peso_total, unit='B', unit_scale=True, unit_divisor=1024,
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
            ) as pbar:
                while True:
                    buf = fsrc.read(1024 * 1024 * 16)
                    if not buf:
                        break
                    fdst.write(buf)
                    os.posix_fadvise(fsrc.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
                    os.posix_fadvise(fdst.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
                    pbar.update(len(buf))
            copiados += 1
    return copiados

print("\n2️⃣ Verificando modelos y LoRAs...")
if drive_connected and os.path.exists(drive_checkpoints):
    copiar_archivos_sin_ram(drive_checkpoints, local_checkpoints)
    copiar_archivos_sin_ram(drive_loras, local_loras)

# 4. Descargas de Modelos Seleccionados
# CyberRealistic XL Play v5.0
if Descargar_CyberRealistic_XL and not os.path.exists(f"{local_checkpoints}/CyberRealisticXLPlay_V5_FP16.safetensors"):
    print("\n🌐 Descargando CyberRealistic XL Play v5.0 (Ultra-Realista / Sin Censura)...")
    !wget -c "https://huggingface.co/cyberdelia/CyberRealisticXL/resolve/main/CyberRealisticXLPlay_V5_FP16.safetensors" -O "{local_checkpoints}/CyberRealisticXLPlay_V5_FP16.safetensors"

# Juggernaut XL v9 Photo
if Descargar_Juggernaut_XL_v9 and not os.path.exists(f"{local_checkpoints}/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors"):
    print("\n🌐 Descargando Juggernaut XL v9 Photo (Hiper-Realismo de Estudio / Texturas)...")
    !wget -c "https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors" -O "{local_checkpoints}/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors"

# RealVisXL V5.0
if Descargar_RealVisXL_v5 and not os.path.exists(f"{local_checkpoints}/RealVisXL_V5.0.safetensors"):
    print("\n🌐 Descargando RealVisXL V5.0 (Fotografía Cruda / Piel Natural)...")
    !wget -c "https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0.safetensors" -O "{local_checkpoints}/RealVisXL_V5.0.safetensors"

# Descargar LoRAs de Realismo
if Descargar_Loras_Realismo:
    if not os.path.exists(f"{local_loras}/add-detail-xl.safetensors"):
        print("\n🌐 Descargando LoRA add-detail-xl...")
        !wget -c "https://huggingface.co/nerfgun3/add-detail-xl/resolve/main/add-detail-xl.safetensors" -O "{local_loras}/add-detail-xl.safetensors"

# Asegurar que al menos un checkpoint existe para arrancar
checkpoints_activos = [f for f in os.listdir(local_checkpoints) if f.endswith(('.safetensors', '.ckpt'))]
if not checkpoints_activos:
    print("\n🌐 Descargando modelo base CyberRealistic XL por defecto...")
    !wget -c "https://huggingface.co/cyberdelia/CyberRealisticXL/resolve/main/CyberRealisticXLPlay_V5_FP16.safetensors" -O "{local_checkpoints}/CyberRealisticXLPlay_V5_FP16.safetensors"

# 5. Dependencias y Limpieza
print("\n3️⃣ Verificando dependencias...")
!pip uninstall -y cupy-cuda12x cupy cupy-cuda11x 2>/dev/null
!pip install --prefer-binary --only-binary=:all: torchsde pytorch_lightning gradio==3.41.2 opencv-contrib-python-headless onnxruntime rembg segment_anything gradio-client==0.5.0 aiofiles ffmpy supervision

# 6. Lanzar Fooocus
print("\n4️⃣ Lanzando Fooocus...")
os.environ["LAUNCH_LIVE_OUTPUT"] = "1"
!python launch.py --share --always-high-vram --disable-preset-download
